In [1]:
import os
os.environ['PYSPARK_PYTHON'] = 'python'
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"
os.environ['HADOOP_HOME'] = 'C:\\hadoop' 
#Start spark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Create Spark session
spark = SparkSession.builder \
    .appName("UserAffinityAnalytics") \
    .master("local[*]") \
    .getOrCreate()

print("Spark session started!")



Spark session started!


In [4]:
df = spark.read.parquet("../data/cleaned_ecommerce_logs.parquet")

print("Dataset loaded!")


Dataset loaded!


In [5]:
df = df.withColumn( #create a new column
    "score",
    F.when(F.col("event_type") == "view", 1) 
     .when(F.col("event_type") == "cart", 3)
     .when(F.col("event_type") == "purchase", 5)
)

print("Dataset with scores:")
df.select("user_id", "category", "event_type", "score").show(10, truncate=False)

Dataset with scores:
+----------+--------+----------+-----+
|user_id   |category|event_type|score|
+----------+--------+----------+-----+
|User_20278|Clothing|cart      |3    |
|User_8521 |Toys    |view      |1    |
|User_12595|Books   |view      |1    |
|User_7858 |Home    |view      |1    |
|User_3315 |Books   |cart      |3    |
|User_23764|Toys    |view      |1    |
|User_17980|Clothing|cart      |3    |
|User_2959 |Books   |view      |1    |
|User_4700 |Toys    |cart      |3    |
|User_17803|Books   |view      |1    |
+----------+--------+----------+-----+
only showing top 10 rows


In [7]:
# Aggregate user preferences
#Reduce and sum up data
#group by userID and Category

user_preferences = df.groupBy("user_id", "category") \
    .agg(F.sum("score").alias("total_score")) # alias renames the coloumn 

print("User preference scores:")

user_preferences.show(10, truncate=False)


User preference scores:
+----------+-----------+-----------+
|user_id   |category   |total_score|
+----------+-----------+-----------+
|User_11424|Toys       |78         |
|User_9072 |Toys       |72         |
|User_18811|Clothing   |63         |
|User_17188|Toys       |99         |
|User_4329 |Home       |73         |
|User_2779 |Home       |96         |
|User_4901 |Home       |99         |
|User_13369|Electronics|102        |
|User_468  |Home       |104        |
|User_24372|Clothing   |80         |
+----------+-----------+-----------+
only showing top 10 rows


In [8]:
# Rank user preferences and format output
# Create a struct so we can sort the categories by score descending for each user
structured_df = user_preferences.withColumn(
    "score_category_struct", 
    F.struct(F.col("total_score"), F.col("category"))
)

# Group by user_id and sort the array of structs descending
grouped_profiles = structured_df.groupBy("user_id").agg(
    F.sort_array(F.collect_list("score_category_struct"), asc=False).alias("sorted_categories")
)

In [9]:
# Transform the sorted structs into the desired array of maps: [{"Category": Score}]
final_profiles = grouped_profiles.withColumn(
    "top_categories",
    F.expr("transform(sorted_categories, x -> map(x.category, x.total_score))")
).drop("sorted_categories")

print("Ranked user profiles:")
final_profiles.show(10, truncate=False)


Ranked user profiles:
+----------+--------------------------------------------------------------------------------------+
|user_id   |top_categories                                                                        |
+----------+--------------------------------------------------------------------------------------+
|User_1    |[{Home -> 101}, {Electronics -> 96}, {Clothing -> 86}, {Toys -> 84}, {Books -> 72}]   |
|User_10   |[{Home -> 98}, {Electronics -> 94}, {Books -> 88}, {Toys -> 84}, {Clothing -> 77}]    |
|User_100  |[{Toys -> 86}, {Books -> 81}, {Clothing -> 75}, {Electronics -> 72}, {Home -> 66}]    |
|User_1000 |[{Home -> 109}, {Electronics -> 99}, {Books -> 94}, {Clothing -> 72}, {Toys -> 71}]   |
|User_10000|[{Toys -> 129}, {Home -> 129}, {Books -> 129}, {Clothing -> 105}, {Electronics -> 87}]|
|User_10001|[{Books -> 102}, {Electronics -> 101}, {Home -> 87}, {Toys -> 81}, {Clothing -> 70}]  |
|User_10002|[{Electronics -> 120}, {Toys -> 102}, {Home -> 94}, {Books -> 89},

In [10]:
# Export affinity 
#Reduced output Dataset 
user_preferences.repartition(48).write.mode("overwrite").parquet(
    "data/affinity_scores"
)

print("Affinity scores exported!")


Affinity scores exported!


In [11]:
# Export ranked user profiles
final_profiles.write.mode("overwrite").json(
    "data/user_profiles"
)

print("User profiles exported!")



User profiles exported!
